In [ ]:
# timm contains many pretrained models including Vision Transformer
!pip install timm -q


In [ ]:
# Basic libraries
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch core libraries
import torch
import torch.nn as nn
import torch.optim as optim

# For loading image datasets
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Pretrained Vision Transformer models
import timm

# Metrics for evaluation
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# For progress bar visualization
from tqdm import tqdm


In [ ]:
# Use GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
# Path to training dataset
train_dir = "/kaggle/input/datasets/javaidahmadwani/lc25000/lung_colon_image_set/Train and Validation Set"

# Path to test dataset
test_dir  = "/kaggle/input/datasets/javaidahmadwani/lc25000/lung_colon_image_set/Test Set"


In [ ]:
# Transformations for training data
# Includes augmentation to improve generalization
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),       # Resize image to 224x224 (ViT input size)
    transforms.RandomHorizontalFlip(),   # Randomly flip images horizontally
    transforms.RandomRotation(10),       # Small rotation for augmentation
    transforms.ToTensor(),               # Convert image to PyTorch tensor
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])  # Normalize using ImageNet stats
])

# Transformations for test data (NO augmentation)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])


In [ ]:
# ImageFolder automatically assigns labels based on folder names
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
test_dataset  = datasets.ImageFolder(test_dir, transform=test_transform)

# Create DataLoaders (batch processing)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Number of classes
num_classes = len(train_dataset.classes)

print("Classes:", train_dataset.classes)
print("Total Classes:", num_classes)


In [ ]:
# Load pretrained Vision Transformer
model = timm.create_model("vit_base_patch16_224", pretrained=True)

# Replace final classification layer with correct number of output classes
model.head = nn.Linear(model.head.in_features, num_classes)

# Move model to GPU/CPU
model = model.to(device)


In [ ]:
# CrossEntropyLoss is used for multi-class classification
criterion = nn.CrossEntropyLoss()

# Adam optimizer updates model weights
optimizer = optim.Adam(model.parameters(), lr=1e-4)


In [ ]:
num_epochs = 50
patience = 5

best_test_loss = float("inf")
patience_counter = 0


In [ ]:
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")

    # ---------------------------
    # TRAINING PHASE
    # ---------------------------
    model.train()  # Set model to training mode
    running_train_loss = 0
    correct_train = 0
    total_train = 0

    train_bar = tqdm(train_loader)

    for images, labels in train_bar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()  # Clear previous gradients

        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)

        loss.backward()  # Backpropagation
        optimizer.step() # Update weights

        running_train_loss += loss.item()

        # Calculate training accuracy
        _, preds = torch.max(outputs, 1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)

    epoch_train_loss = running_train_loss / len(train_loader)
    epoch_train_acc = correct_train / total_train

    train_losses.append(epoch_train_loss)
    train_accuracies.append(epoch_train_acc)

    # ---------------------------
    # TESTING PHASE
    # ---------------------------
    model.eval()  # Set model to evaluation mode
    running_test_loss = 0
    correct_test = 0
    total_test = 0

    with torch.no_grad():  # Disable gradient calculation
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_test_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct_test += (preds == labels).sum().item()
            total_test += labels.size(0)

    epoch_test_loss = running_test_loss / len(test_loader)
    epoch_test_acc = correct_test / total_test

    test_losses.append(epoch_test_loss)
    test_accuracies.append(epoch_test_acc)

    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
    print(f"Test  Loss: {epoch_test_loss:.4f} | Test  Acc: {epoch_test_acc:.4f}")

    # ---------------------------
    # EARLY STOPPING
    # ---------------------------
    if epoch_test_loss < best_test_loss:
        best_test_loss = epoch_test_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_vit_model.pth")
        print("Model improved and saved.")
    else:
        patience_counter += 1
        print("No improvement. Patience:", patience_counter)

        if patience_counter >= patience:
            print("Early stopping triggered.")
            break


In [ ]:
model.load_state_dict(torch.load("best_vit_model.pth"))
model.eval()


In [ ]:
all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)

        _, preds = torch.max(outputs, 1)

        all_labels.extend(labels.numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8,8))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=train_dataset.classes,
            yticklabels=train_dataset.classes)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

print(classification_report(all_labels, all_preds))


In [ ]:
all_labels_bin = label_binarize(all_labels, classes=range(num_classes))
all_probs = np.array(all_probs)

plt.figure()

for i in range(num_classes):
    fpr, tpr, _ = roc_curve(all_labels_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{train_dataset.classes[i]} (AUC = {roc_auc:.3f})")

plt.plot([0,1],[0,1],'--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multi-Class ROC Curve")
plt.legend()
plt.show()


In [ ]:
plt.figure()
plt.plot(train_losses, label="Train Loss")
plt.plot(test_losses, label="Test Loss")
plt.legend()
plt.title("Loss Curve")
plt.show()

plt.figure()
plt.plot(train_accuracies, label="Train Accuracy")
plt.plot(test_accuracies, label="Test Accuracy")
plt.legend()
plt.title("Accuracy Curve")
plt.show()
